### ¿Cuál es el precio promedio de los alojamientos por barrio y distrito?

In [7]:
import os
from sqlalchemy import create_engine, text
import pandas as pd
from tabulate import tabulate
from dotenv import load_dotenv

# CARGAR EL ARCHIVO .ENV
load_dotenv() 

# Recuperamos las variables
DB_USER = os.getenv("MYSQL_USER")
DB_PASSWORD = os.getenv("MYSQL_PASSWORD")
DB_HOST = os.getenv("MYSQL_HOST")
DB_PORT = os.getenv("MYSQL_PORT", "3306")
DB_NAME = "core"

DATABASE_URL = f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

# 2. Crear el motor (engine)
engine = create_engine(DATABASE_URL)

with engine.connect() as connection:
            # Ejemplo: SELECT
            query = text('''SELECT dn.neighbourhood , dn.neighbourhood_group , avg(price)
            FROM core.fact_listings fl
            INNER JOIN core.dim_neighbourhoods dn
            ON fl.id_neighbourhood = dn.id_neighbourhood 
            GROUP BY dn.neighbourhood , dn.neighbourhood_group
            order by 1,2''')
            result = connection.execute(query)
            
            # Cargamos el resultado directamente en un DataFrame
            df = pd.read_sql(query, engine)

            print("\n--- RESULTADOS DE LA TABLA ---")
            print(tabulate(df, headers='keys', tablefmt='psql', showindex=False))


--- RESULTADOS DE LA TABLA ---
+----------------------------+-----------------------+--------------+
| neighbourhood              | neighbourhood_group   |   avg(price) |
|----------------------------+-----------------------+--------------|
| Allerton                   | Bronx                 |      90.5946 |
| Arden Heights              | Staten Island         |      67.25   |
| Arrochar                   | Staten Island         |     118.25   |
| Arverne                    | Queens                |     158.515  |
| Astoria                    | Queens                |     116.018  |
| Bath Beach                 | Brooklyn              |      84.8    |
| Battery Park City          | Manhattan             |     182.194  |
| Bay Ridge                  | Brooklyn              |     105.374  |
| Bay Terrace                | Queens                |     118.8    |
| Bay Terrace, Staten Island | Staten Island         |     102.5    |
| Baychester                 | Bronx                 |    

---

### ¿Qué tipo de habitación es el más ofrecido y cuál genera mayor revenue estimado?

In [9]:
with engine.connect() as connection:

            query = text('''SELECT room_type, count(*), SUM(revenue) FROM 
            (
            SELECT drt.room_type, fl.price * fl.minimum_nights as revenue 
            FROM core.fact_listings fl
            INNER JOIN core.dim_room_types drt 
            ON fl.id_room_type = drt.id_room_type 
            ) sub
            GROUP BY room_type
            order by 2 desc''')
            result = connection.execute(query)
            
            # Cargamos el resultado directamente en un DataFrame
            df = pd.read_sql(query, engine)

            print("\n--- RESULTADOS DE LA TABLA ---")
            print(tabulate(df, headers='keys', tablefmt='psql', showindex=False, floatfmt=",.0f"))


--- RESULTADOS DE LA TABLA ---
+-----------------+------------+----------------+
| room_type       |   count(*) |   SUM(revenue) |
|-----------------+------------+----------------|
| Entire home/apt |      20332 |     27,143,982 |
| Private room    |      17665 |      8,486,321 |
| Shared room     |        846 |        200,652 |
+-----------------+------------+----------------+


CONCLUSION: La habitacion mas ofrecida y que genera mayor revenue es del tipo "Entire home/apt"

---

### ¿Cuáles son los anfitriones con más propiedades listadas y cómo varían sus precios?

In [10]:
with engine.connect() as connection:

            query = text('''SELECT host_id, host_name, count(*), MAX(price), MIN(price)
FROM core.fact_listings fl
GROUP BY host_id, host_name
order by count(*) desc
limit 10''')
            result = connection.execute(query)
            
            # Cargamos el resultado directamente en un DataFrame
            df = pd.read_sql(query, engine)

            print("\n--- RESULTADOS DE LA TABLA ---")
            print(tabulate(df, headers='keys', tablefmt='psql', showindex=False, floatfmt=",.0f"))


--- RESULTADOS DE LA TABLA ---
+-----------+-------------------+------------+--------------+--------------+
|   host_id | host_name         |   count(*) |   MAX(price) |   MIN(price) |
|-----------+-------------------+------------+--------------+--------------|
| 219517861 | Sonder (NYC)      |        207 |          616 |          100 |
|  61391963 | Corporate Housing |         79 |          200 |          109 |
|  16098958 | Jeremy & Laura    |         61 |          400 |          117 |
| 137358866 | Kazuya            |         51 |           76 |           31 |
|   7503643 | Vida              |         49 |          199 |          129 |
| 190921808 | John              |         46 |          650 |           45 |
|  30283594 | Kara              |         43 |          479 |          109 |
|   1475015 | Mike              |         42 |          150 |           83 |
| 120762452 | Stanley           |         40 |          367 |          111 |
|   2119276 | Host              |         39

CONCLUSION: El anfitrion con mas propiedades listadas es 'Sonder (NYC)" y tiene una diferencia de U$516 entre su precio maximo y minimo

---

### ¿Existen diferencias significativas en la disponibilidad anual entre barrios o tipos de alojamiento?

In [11]:
with engine.connect() as connection:

            query = text('''SELECT neighbourhood, total_availability
FROM (
    SELECT 
        dn.neighbourhood, 
        SUM(fl.availability_365) AS total_availability,
        RANK() OVER (ORDER BY SUM(fl.availability_365) DESC) as ranking_max,
        RANK() OVER (ORDER BY SUM(fl.availability_365) ASC) as ranking_min
    FROM core.fact_listings fl
    LEFT JOIN core.dim_neighbourhoods dn
	ON fl.id_neighbourhood = dn.id_neighbourhood
    GROUP BY fl.id_neighbourhood
) as subquery
WHERE ranking_max = 1 OR ranking_min = 1;''')
            result = connection.execute(query)
            
            # Cargamos el resultado directamente en un DataFrame
            df = pd.read_sql(query, engine)

            print("\n--- RESULTADOS DE LA TABLA ---")
            print(tabulate(df, headers='keys', tablefmt='psql', showindex=False, floatfmt=",.0f"))


--- RESULTADOS DE LA TABLA ---
+----------------------------+----------------------+
| neighbourhood              |   total_availability |
|----------------------------+----------------------|
| Bay Terrace, Staten Island |                    0 |
| Bedford-Stuyvesant         |              389,061 |
+----------------------------+----------------------+


CONCLUSION: Hay diferencia muy significativa en la disponibilidad anual entre el barrio con mayor  y el de menor disponibilidad.

In [12]:
with engine.connect() as connection:

            query = text('''SELECT room_type, total_availability
FROM (
    SELECT 
        drt.room_type, 
        SUM(fl.availability_365) AS total_availability,
        RANK() OVER (ORDER BY SUM(fl.availability_365) DESC) as ranking_max,
        RANK() OVER (ORDER BY SUM(fl.availability_365) ASC) as ranking_min
    FROM core.fact_listings fl
    LEFT JOIN core.dim_room_types drt 
	ON fl.id_room_type = drt.id_room_type
    GROUP BY fl.id_room_type
) as subquery
WHERE ranking_max = 1 OR ranking_min = 1;''')
            result = connection.execute(query)
            
            # Cargamos el resultado directamente en un DataFrame
            df = pd.read_sql(query, engine)

            print("\n--- RESULTADOS DE LA TABLA ---")
            print(tabulate(df, headers='keys', tablefmt='psql', showindex=False, floatfmt=",.0f"))


--- RESULTADOS DE LA TABLA ---
+-----------------+----------------------+
| room_type       |   total_availability |
|-----------------+----------------------|
| Shared room     |              140,435 |
| Entire home/apt |            2,264,454 |
+-----------------+----------------------+


CONCLUSION: Hay diferencia muy significativa en la disponibilidad anual entre el tipo de alojamiento con mayor y el de menor disponibilidad.

---

### ¿Qué barrios tienen la mayor concentración de alojamientos activos?

In [13]:
with engine.connect() as connection:

            query = text('''SELECT dn.neighbourhood, count(*) cantidad_activo
FROM core.fact_listings fl
LEFT JOIN core.dim_neighbourhoods dn
ON fl.id_neighbourhood = dn.id_neighbourhood
GROUP BY fl.id_neighbourhood
order by count(*) desc
limit 10''')
            result = connection.execute(query)
            
            # Cargamos el resultado directamente en un DataFrame
            df = pd.read_sql(query, engine)

            print("\n--- RESULTADOS DE LA TABLA ---")
            print(tabulate(df, headers='keys', tablefmt='psql', showindex=False, floatfmt=",.0f"))


--- RESULTADOS DE LA TABLA ---
+--------------------+-------------------+
| neighbourhood      |   cantidad_activo |
|--------------------+-------------------|
| Williamsburg       |              3163 |
| Bedford-Stuyvesant |              3141 |
| Harlem             |              2206 |
| Bushwick           |              1944 |
| Hell's Kitchen     |              1532 |
| East Village       |              1490 |
| Upper West Side    |              1482 |
| Upper East Side    |              1405 |
| Crown Heights      |              1265 |
| Midtown            |               986 |
+--------------------+-------------------+


CONCLUSION: Top 10 de barrios con mayor cantidad de alojamientos activos, siendo el de mayor ocupacion "Williamsburg"